# Виртуальный анализатор: бутан в кубе дебутанизатора (прототип)

Исследовательский ноутбук. Цель — по технологическим тегам колонны предсказать
содержание C4 (бутана) в кубовом продукте, которое лаборатория меряет редко и с задержкой.

Входы (как в эталонном датасете Fortuna et al., 2007):
u1 — температура верха, u2 — давление верха, u3 — расход флегмы,
u4 — расход на следующую установку, u5 — температура 6-й тарелки,
u6, u7 — температура низа (два датчика). Выход y — содержание бутана.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

## Данные
Реальные данные заказчика ещё не пришли, генерю похожие.

In [ ]:
np.random.seed(42)
N = 2394
def ar(n, phi, s):
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi*x[i-1] + np.random.randn()*s
    return x
u1 = 0.5 + ar(N, 0.98, 0.02)
u2 = 0.5 + ar(N, 0.97, 0.015)
u3 = 0.5 + ar(N, 0.95, 0.03)
u4 = 0.5 + ar(N, 0.96, 0.025)
u5 = 0.5 + ar(N, 0.985, 0.02)
u6 = 0.5 + ar(N, 0.99, 0.015)
u7 = u6 + np.random.randn(N)*0.01
y = np.zeros(N)
for t in range(6, N):
    y[t] = 0.3 - 0.8*(u5[t-4]-0.5) - 0.6*(u6[t-2]-0.5) + 0.35*(u3[t-1]-0.5) + 0.9*(u5[t-4]-0.5)**2 + 0.2*(u2[t]-0.5)*(u1[t]-0.5)*4 + np.random.randn()*0.015
y = np.clip(y, 0, 1)
df = pd.DataFrame({'u1':u1,'u2':u2,'u3':u3,'u4':u4,'u5':u5,'u6':u6,'u7':u7,'y':y})
df.to_csv('../data/debutanizer.csv', index=False)
df.head()

In [ ]:
df = pd.read_csv('../data/debutanizer.csv')
df.describe()

In [ ]:
df.plot(subplots=True, figsize=(12,14))
plt.show()

## Признаки
Процесс инерционный — добавляю лаги.

In [ ]:
for c in ['u1','u2','u3','u4','u5','u6','u7']:
    for l in [1,2,3,4,5,6]:
        df[c+'_lag'+str(l)] = df[c].shift(l)
df['u6u7'] = (df['u6']+df['u7'])/2
df = df.dropna()
X = df.drop('y', axis=1)
y = df['y']

In [ ]:
# train/test по времени
n = int(len(X)*0.8)
X_train = X[:n]
X_test = X[n:]
y_train = y[:n]
y_test = y[n:]
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

## Модели

In [ ]:
m1 = Ridge(alpha=1.0)
m1.fit(X_train_s, y_train)
p1 = m1.predict(X_test_s)
print('Ridge RMSE', np.sqrt(mean_squared_error(y_test, p1)), 'R2', r2_score(y_test, p1), 'MAE', mean_absolute_error(y_test,p1))

In [ ]:
m2 = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=42)
m2.fit(X_train_s, y_train)
p2 = m2.predict(X_test_s)
print('HGB RMSE', np.sqrt(mean_squared_error(y_test, p2)), 'R2', r2_score(y_test, p2), 'MAE', mean_absolute_error(y_test,p2))

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(y_test.values, label='lab')
plt.plot(p1, label='ridge')
plt.plot(p2, label='hgb')
plt.legend(); plt.show()

## Сохранение
Бустинг лучше — сохраняю его вместе со скейлером.

In [ ]:
pickle.dump((sc, m2), open('../models/model_final_v2_new.pkl', 'wb'))

In [ ]:
# применение на новых данных
sc, m = pickle.load(open('../models/model_final_v2_new.pkl','rb'))
new = pd.read_csv('../data/debutanizer.csv').tail(50)
for c in ['u1','u2','u3','u4','u5','u6','u7']:
    for l in [1,2,3,4,5,6]:
        new[c+'_lag'+str(l)] = new[c].shift(l)
new['u6u7'] = (new['u6']+new['u7'])/2
new = new.dropna()
print(m.predict(sc.transform(new.drop('y',axis=1))))